# Add Zones to Silver

## Import Packages

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    BooleanType,
    DateType,
    TimestampType,
    DecimalType
)
import requests
from decimal import Decimal

In [0]:
from pyspark import pipelines as dp

## Variables

In [0]:
nyc_zone_data_api = "https://data.cityofnewyork.us/resource/8meu-9t5y.json"

## Schema Definition

In [0]:
api_schema = StructType(
    [
        StructField(
            name="the_geom",
            dataType=StringType(),
        ),
        StructField(
            name="shape_leng",
            dataType=DecimalType(30,20),
        ),
        StructField(
            name="shape_area",
            dataType=DecimalType(30,20),
        ),
        StructField(
            name="zone",
            dataType=StringType(),
        ),
        StructField(
            name="locationid",
            dataType=IntegerType(),
        ),
        StructField(
            name="borough",
            dataType=StringType(),
        ),
    ]
)

In [0]:
schema = StructType(
    [
        StructField(
            name="location_id",
            dataType=IntegerType(),
            nullable=False,
            metadata={"comment":"Shows the Location ID of each NYC Zone"}
        ),
        StructField(
            name="borough",
            dataType=StringType(),
            nullable=False,
            metadata={"comment":"Shows the Borough Name"}
        ),
        StructField(
            name="zone",
            dataType=StringType(),
            nullable=False,
            metadata={"comment":"Shows the Zone Name"}
        ),
        StructField(
            name="service_zone",
            dataType=StringType(),
            nullable=False,
            metadata={"comment":"Shows the Service Zone Name"}
        ),
        StructField(
            name="the_geom",
            dataType=StringType(),
            nullable=True,
            metadata={"comment": "Presents the Location as Polygon Data for a Shape Map"}
        ),
        StructField(
            name="shape_leng",
            dataType=DecimalType(30,20),
            nullable=True,
            metadata={"comment": "Shows the Length of the Shape"}
        ),
        StructField(
            name="shape_area",
            dataType=DecimalType(30,20),
            nullable=True,
            metadata={"comment": "Shows the Area of the Shape"}
        ),
        # ToDo maybe add Lon and Lat
    ]
)

## ETL

In [0]:
response = requests.get(nyc_zone_data_api)
response.raise_for_status()

data = response.json()

# Convert string numeric fields to appropriate types
for record in data:
    if 'shape_leng' in record and isinstance(record['shape_leng'], str):
        record['shape_leng'] = Decimal(record['shape_leng'])
    if 'shape_area' in record and isinstance(record['shape_area'], str):
        record['shape_area'] = Decimal(record['shape_area'])
    if 'locationid' in record and isinstance(record['locationid'], str):
        record['locationid'] = int(record['locationid'])

df_zone_data = spark.createDataFrame(data=data, schema=api_schema)
 
# Fix the wrong API data Handwritten changes not future prove but at the moment fastest way
# Id 56 exists twice and should be 56 / 57
df_zone_data = df_zone_data.withColumn("locationid", F.when((F.col("locationid") == 56) & (F.col("shape_area") < 0.0001 ), 57).otherwise(F.col("locationid")))

# Id 103 esists three times and should be 103 / 104 / 105
df_zone_data = df_zone_data.withColumn("locationid", F.when((F.col("locationid") == 103) & (F.col("shape_area") < 0.0003 ) & (F.col("shape_area") > 0.00001), 104).otherwise(F.col("locationid")))
df_zone_data = df_zone_data.withColumn("locationid", F.when((F.col("locationid") == 103) & (F.col("shape_area") >= 0.0003 ), 105).otherwise(F.col("locationid")))

# Fix GeoJson Data
df_zone_data = df_zone_data.withColumn(
    "the_geom",
    F.regexp_replace(F.col("the_geom"), "'", '"')
)
df_zone_data = df_zone_data.filter(F.col("the_geom").isNotNull())

In [0]:
@dp.materialized_view(
    # Name der Zieltabelle
    name="analytics.silver.geographics_stm_zone",
    # Beschreibung der Tabelle
    comment="This table shows the zones in NYC and their Lon and Lat Values",
    # Liquid Clustering (Statt partitioning und Z-Order)
    cluster_by=[],
    cluster_by_auto=True,
    schema=schema,
)
def geographics_stm_zone():
    df = spark.read.table("analytics.bronze.dwh_nyc_taxi_zone")
    df = df.withColumnRenamed("LocationID", "location_id")
    df = df.dropDuplicates(["location_id"])

    # Join in Geodata
    df = df.join(
        other=df_zone_data,
        on=df["location_id"] == df_zone_data["locationid"],
        how="left",
    ).select(
        df["*"],
        df_zone_data["the_geom"].alias("the_geom"),
        df_zone_data["shape_leng"].alias("shape_leng"),
        df_zone_data["shape_area"].alias("shape_area"),
    )

    schema_columns = [(field.name, field.dataType) for field in schema.fields]
    df = df.select([F.col(col_name).cast(col_dtype) for col_name, col_dtype in schema_columns])
        
    return df